# 03 Agentic App: End-to-End Support Workflow

This notebook demonstrates the complete CultPass support-ticket workflow using the real LangGraph orchestrator, OpenAI LLM, SQLite data, Chroma knowledge base, and DB/RAG tool logic. It uses in-process tool adapters so saved notebook execution is stable while the CLI app continues to load the same tools over MCP transport.

## Setup

In [1]:
from __future__ import annotations

import io
import json
import logging
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "solution" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from solution.agentic.tools.local_tools import load_local_tools
from solution.agentic.workflow import build_orchestrator
from solution.logging_config import LOG_FORMAT, configure_logging

load_dotenv(PROJECT_ROOT / ".env")
configure_logging()

c:\Users\rsurs\OneDrive\Documents\Root\project-4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WindowsPath('C:/Users/rsurs/OneDrive/Documents/Root/project-4/logs/udahub.log')

In [2]:
log_stream = io.StringIO()
log_handler = logging.StreamHandler(log_stream)
log_handler.setFormatter(logging.Formatter(LOG_FORMAT))

udahub_logger = logging.getLogger("udahub")
udahub_logger.setLevel(logging.INFO)
udahub_logger.addHandler(log_handler)
udahub_logger.propagate = False

print("Timestamped log capture is ready for udahub.* loggers.")

Timestamped log capture is ready for udahub.* loggers.


In [3]:
orchestrator = build_orchestrator(load_local_tools())
print("Real orchestrator loaded with local DB/RAG tool adapters and OpenAI LLM.")

Real orchestrator loaded with local DB/RAG tool adapters and OpenAI LLM.


## Trace Helper

In [4]:
async def run_ticket(ticket_id: str, customer_message: str) -> dict:
    log_stream.seek(0)
    log_stream.truncate(0)

    config = {"configurable": {"thread_id": ticket_id}}
    state = await orchestrator.ainvoke(
        {"ticket_id": ticket_id, "messages": [HumanMessage(content=customer_message)]},
        config=config,
    )

    retrieved_docs = state.get("retrieved_docs") or []
    tool_results = state.get("tool_results") or []
    final_message = state["messages"][-1].content if state.get("messages") else ""
    logs = log_stream.getvalue().strip()

    print(f"TICKET: {ticket_id}")
    print(f"USER: {customer_message}")
    print(f"CLASSIFICATION: category={state.get('category')} priority={state.get('priority')} confidence={state.get('confidence')}")
    print(f"ROUTE: {state.get('route')}")
    print(f"RETRIEVED_DOCS: count={len(retrieved_docs)} titles={[doc.get('title') for doc in retrieved_docs]}")
    print("TOOL_RESULTS:")
    print(json.dumps(tool_results, indent=2, default=str) if tool_results else "[]")
    print(f"ESCALATION_REQUIRED: {state.get('escalation_required')}")
    print("FINAL_RESPONSE:")
    print(final_message)
    print("LOGS:")
    print(logs)

    return state

## Scenario 1: Knowledge-Base Resolution

In [5]:
kb_state = await run_ticket(
    "demo-kb-resolution",
    "How do I reserve an event on CultPass?",
)

2026-06-17 17:15:01,073 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-17 17:15:02,640 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:03,635 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-17 17:15:05,932 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:06,427 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


TICKET: demo-kb-resolution
USER: How do I reserve an event on CultPass?
CLASSIFICATION: category=General priority=low confidence=0.9
ROUTE: resolver
RETRIEVED_DOCS: count=0 titles=[]
TOOL_RESULTS:
[]
ESCALATION_REQUIRED: True
FINAL_RESPONSE:
Hi there! Thank you for reaching out to us. I understand you have a question about reserving an event on CultPass. A member of our team will follow up with you shortly to assist you further!
LOGS:
2026-06-17 17:15:00,330 INFO udahub.tools.vector_store: Opening Chroma collection 'udahub_memory_openai' at C:\Users\rsurs\OneDrive\Documents\Root\project-4\solution\data\index
2026-06-17 17:15:01,075 INFO udahub.supervisor: Supervisor loaded 0 memories for demo-kb-resolution
2026-06-17 17:15:02,657 INFO udahub.classifier: Classified as General (p=low, conf=0.90)
2026-06-17 17:15:02,657 INFO udahub.classifier: Classifier routed to resolver
2026-06-17 17:15:02,657 INFO udahub.tools.vector_store: Opening Chroma collection 'cultpass_knowledge_openai' at C:\U

## Scenario 2: Tool-Assisted Account Lookup

In [6]:
tool_state = await run_ticket(
    "demo-tool-subscription",
    "What subscription do I have? My customer id is f556c0.",
)

2026-06-17 17:15:06,933 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-17 17:15:09,169 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:11,553 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:13,807 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:14,284 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


TICKET: demo-tool-subscription
USER: What subscription do I have? My customer id is f556c0.
CLASSIFICATION: category=Membership priority=medium confidence=0.9
ROUTE: tool_agent
RETRIEVED_DOCS: count=0 titles=[]
TOOL_RESULTS:
[
  {
    "tool": "get_subscription",
    "result": {
      "found": true,
      "user_id": "f556c0",
      "status": "active",
      "tier": "premium",
      "monthly_quota": 5
    }
  }
]
ESCALATION_REQUIRED: False
FINAL_RESPONSE:
You have an active **premium** subscription with a monthly quota of **5**. If you have any more questions or need assistance, feel free to ask!
LOGS:
2026-06-17 17:15:06,458 INFO udahub.tools.vector_store: Opening Chroma collection 'udahub_memory_openai' at C:\Users\rsurs\OneDrive\Documents\Root\project-4\solution\data\index
2026-06-17 17:15:06,933 INFO udahub.supervisor: Supervisor loaded 0 memories for demo-tool-subscription
2026-06-17 17:15:09,182 INFO udahub.classifier: Classified as Membership (p=medium, conf=0.90)
2026-06-17 17:15

## Scenario 3: Escalation After Tool Usage

In [7]:
escalation_state = await run_ticket(
    "demo-refund-escalation",
    "Please approve a refund for customer a4ab87 for my last booking.",
)

2026-06-17 17:15:14,738 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-17 17:15:16,006 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:17,158 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:18,855 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:20,111 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:23,312 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:25,643 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:26,149 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


TICKET: demo-refund-escalation
USER: Please approve a refund for customer a4ab87 for my last booking.
CLASSIFICATION: category=Refund priority=medium confidence=0.9
ROUTE: tool_agent
RETRIEVED_DOCS: count=0 titles=[]
TOOL_RESULTS:
[
  {
    "tool": "lookup_customer",
    "result": {
      "found": true,
      "user_id": "Alice Kingsley",
      "email": "alice.kingsley@wonderland.com",
      "is_blocked": true,
      "subscription": {
        "status": "cancelled",
        "tier": "basic",
        "monthly_quota": 3
      }
    }
  },
  {
    "tool": "list_reservations",
    "result": {
      "user_id": "a4ab87",
      "count": 2,
      "reservation": [
        {
          "reservation_id": "346e63",
          "status": "reserved",
          "experience_title": "Carnival History Tour in Olinda",
          "when": "2026-06-14T14:33:44.619594",
          "is_premium": true
        },
        {
          "reservation_id": "4f8fb9",
          "status": "reserved",
          "experience_titl

## Scenario 4: Edge Case / Unknown Issue

In [8]:
edge_state = await run_ticket(
    "demo-edge-unknown",
    "Can CultPass arrange a private moon concert next week?",
)

2026-06-17 17:15:26,610 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-17 17:15:27,795 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:28,251 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-17 17:15:31,198 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-17 17:15:31,648 INFO httpx: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


TICKET: demo-edge-unknown
USER: Can CultPass arrange a private moon concert next week?
CLASSIFICATION: category=General priority=low confidence=0.9
ROUTE: resolver
RETRIEVED_DOCS: count=0 titles=[]
TOOL_RESULTS:
[]
ESCALATION_REQUIRED: True
FINAL_RESPONSE:
Hi there! Thank you for reaching out to us about arranging a private moon concert. I understand how exciting this is, and I want to assure you that a member of our team will follow up with you shortly to discuss the details.
LOGS:
2026-06-17 17:15:26,186 INFO udahub.tools.vector_store: Opening Chroma collection 'udahub_memory_openai' at C:\Users\rsurs\OneDrive\Documents\Root\project-4\solution\data\index
2026-06-17 17:15:26,610 INFO udahub.supervisor: Supervisor loaded 0 memories for demo-edge-unknown
2026-06-17 17:15:27,798 INFO udahub.classifier: Classified as General (p=low, conf=0.90)
2026-06-17 17:15:27,798 INFO udahub.classifier: Classifier routed to resolver
2026-06-17 17:15:27,811 INFO udahub.tools.vector_store: Opening Chrom